# Notebook for synthesizing audio from text
[Google Cloud Platform Text to Speech](https://cloud.google.com/text-to-speech/)  
See [Google AI Blog - Tacotron 2](https://ai.googleblog.com/2017/12/tacotron-2-generating-human-like-speech.html)

In [19]:
import os
import pathlib
from tqdm import tqdm
from pydub import AudioSegment
from google.cloud import texttospeech

## Process text file

In [2]:
all_raw_convos = []
with open('2021_04_15_gpt2_output_topk-40_1558.txt', 'r') as infile:
    raw_in = infile.read()
    
for line in raw_in.split('||'):
    all_raw_convos.append(line.strip())
    
all_convos = []
for raw_convo in all_raw_convos:
    one_convo = []
    for raw_line in raw_convo.split('\n'):
        if raw_line != '':
            one_convo.append(raw_line.strip())
    all_convos.append(one_convo)


In [3]:
len(all_convos)

181

In [4]:
all_convos[100][2]

"As I have already addressed in this post, the economic expansion since 2008 was primarily fueled by the expansion of household wealth. However, households' net worth has not recovered to where it was in the late 1990s, as more than 80 percent of households have experienced a declining net worth, while the richest 10 percent has more money than they have ever had since the Depression. At the same time, the US economy has grown more than 30 percent since 2009, with jobs creation and productivity growth being the most important factors."

In [5]:
total = 0
for convo in all_convos:
    for sentence in convo:
        total += 1
print(total)

1661


## Set voice parameters

In [6]:
credential_path = "/home/dave/projects/SEARCHLIGHT/VTC/GCP_text_to_speech/5e41ca424778.json"
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = credential_path

In [7]:
# Instantiates a client
client = texttospeech.TextToSpeechClient()

In [8]:
# Set the text input to be synthesized
synthesis_input = texttospeech.SynthesisInput(text="This is a test of the Google Cloud text to speech engine for VTC dialog generation.")

Select Voice name, e.g. "en-US-Wavenet-F"

In [21]:
# Build the voice request, select the language code ("en-US") and the ssml
# voice gender ("neutral")
voice = texttospeech.VoiceSelectionParams(
    language_code="en-US", ssml_gender=texttospeech.SsmlVoiceGender.NEUTRAL, name="en-US-Wavenet-H"
)

In [22]:
# Select the type of audio file you want returned
audio_config = texttospeech.AudioConfig(
    audio_encoding=texttospeech.AudioEncoding.LINEAR16
)

## Speech Generation Loop

In [25]:
path_root = "/home/dave/projects/SEARCHLIGHT/VTC/VTC_audio_tracks"

In [26]:
voice_name = "/female_1-en-US-Wavenet-H"
path_root += voice_name

In [27]:
# First 25 convos
for x in tqdm(range(25)):
    convo_path = path_root + "/convo_" + str(x) +'/'
    pathlib.Path(convo_path).mkdir(parents=True, exist_ok=True) 
    for y in range(len(all_convos[x])):
        filename = "line_" + str(y) + ".flac"
        # Generate speech
        synthesis_input = texttospeech.SynthesisInput(text=all_convos[x][y])
        response = client.synthesize_speech(input=synthesis_input, voice=voice, audio_config=audio_config)
        # write file
        audio_output = AudioSegment(response.audio_content)
        audio_output.export(convo_path + filename, format="flac")
        

100%|██████████| 25/25 [01:54<00:00,  4.57s/it]
